# 03 · Retrieval — Query 技巧（Rewrite / Multi-Query / HyDE）

目标：
- 复用课件里的“Query 改写”思路
- 在同一个 Chroma collection 上对比：
  - baseline（原始 query）
  - query rewrite（LLM 改写）
  - multi-query（生成多条查询，融合召回）
  - HyDE（先生成假设答案，再用它去检索）

> 依赖：上一节已写入 `data/chroma` + 设置 `OPENAI_API_KEY`。


In [4]:
pip install langchain_openai langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.48-cp311-cp311-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchain_classic-1.0.2-py3-none-any.whl (1.0 MB)
Using cached la

In [8]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL", "text-embedding-3-small")
chat_model = os.getenv("CHAT_MODEL") or os.getenv("LLM_MODEL", "openai/gpt-4o")

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,



}

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先重建 data/chroma。
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
print("ready:", COLLECTION, "embed:", embed_model, "chat:", chat_model, "env:", ENV_FILE)


ready: autel_annual_report_2024 embed: text-embedding-3-small chat: openai/gpt-4o env: /Users/mengbai/Documents/AI-training/.env


In [9]:
# baseline 检索


QUESTION = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？给出相关表格或段落。"

hits = vs.similarity_search(QUESTION, k=5)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:350].replace("\n", " "))
    print()


[1] {'parse_source': 'paddleocr_vl', 'file_name': '道通24年年报_p0051-0100_3.md', 'chunk_in_file': 3, 'doc_group': '道通24年年报_p0051-0100', 'doc_id': '道通24年年报_p0051-0100_3', 'source': '道通24年年报', 'h2': '2、收入和成本分析', 'file_path': 'paddleocr_vl/道通24年年报_p0051-0100/道通24年年报_p0051-0100_3.md', 'chunk_id': 344}
break-word;'>37.27</td><td style='text-align: center; word-wrap: break-word;'>52.98</td><td style='text-align: center; word-wrap: break-word;'>44.66</td><td style='text-align: center; word-wrap: break-word;'>增加3.62个百分点</td></tr><tr><td colspan="7">主营业务分产品情况</td></tr><tr><td style='text-align: center; word-wrap: break-word;'>分产品</td><td style='text-

[2] {'file_name': '道通24年年报_p0251-0300_40.md', 'doc_group': '道通24年年报_p0251-0300', 'source': '道通24年年报', 'chunk_in_file': 5, 'chunk_id': 1874, 'parse_source': 'paddleocr_vl', 'h2': '6、分部信息', 'doc_id': '道通24年年报_p0251-0300_40', 'h3': '(2). 报告分部的财务信息', 'file_path': 'paddleocr_vl/道通24年年报_p0251-0300/道通24年年报_p0251-0300_40.md'}
<table border=1 style='margin: au

In [10]:
# 1) Query Rewrite：把“口语/模糊问法”改成更利于召回的检索式问法

from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是检索系统的 Query Rewrite 模块。输出 1 条更利于文档检索的中文查询句，不要回答问题。",
        ),
        ("human", "原始问题：{q}"),
    ]
)

rewritten = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
print("rewritten:\n", rewritten)

hits = vs.similarity_search(rewritten, k=5)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


rewritten:
 道通2024年年报主营业务和产品线收入结构分析
[1] {'h3': '1. 事项描述', 'file_path': 'paddleocr_vl/道通24年年报_p0101-0150/道通24年年报_p0101-0150_42.md', 'chunk_id': 797, 'doc_id': '道通24年年报_p0101-0150_42', 'h2': '(一) 收入确认', 'source': '道通24年年报', 'parse_source': 'paddleocr_vl', 'doc_group': '道通24年年报_p0101-0150', 'chunk_in_file': 4, 'file_name': '道通24年年报_p0101-0150_42.md'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'source': '道通24年年报', 'chunk_id': 35, 'h3': '一、经营情况讨论与分析', 'doc_id': '道通24年年报_p0001-0050_14', 'file_path': 'paddleocr_vl/道通24年年报_p0001-0050/道通24年年报_p0001-0050_14.md', 'doc_group': '道通24年年报_p0001-0050', 'chunk_in_file': 0, 'file_name': '道通24年年报_p0001-0050_14.md', 'parse_source': 'paddleocr_vl'}
长期以来，公司以 AI 为核心驱动力，紧密围绕 “智能化” 战略布局业务生态，不断推动 AI 技术与业务的深度融合。   2024 年，道通科技 “全面 AI”，加速推动 AI 技术与业务场景和组织变革深度融合，进一步巩固数字维修的全球领导者地位，致力于成为智慧能源领域的全球领军企业以及空地一体集群智慧解决方案的全球领导者，努力成为 AI 行

In [11]:
# 2) Multi-Query：生成多条“角度不同但等价”的查询，提升召回覆盖率

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Multi-Query 生成器。给定一个问题，输出 4 条可用于检索的中文 query，每条一行，不要编号，不要解释。",
        ),
        ("human", "问题：{q}"),
    ]
)

queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
print("queries:")
for q in queries:
    print("-", q)

# 融合策略：简单 union（课堂演示足够；生产可用 RRF）
seen = set()
merged = []
for q in queries:
    for d in vs.similarity_search(q, k=4):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        if key not in seen:
            merged.append(d)
            seen.add(key)

print("merged hits:", len(merged))
for i, d in enumerate(merged[:8], 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


queries:
- 道通2024年年报主营业务收入结构
- 道通2024年年报产品线收入表格
- 道通2024年年报收入结构段落
- 道通2024年年报主营产品收入分析
merged hits: 10
[1] {'doc_id': '道通24年年报_p0101-0150_42', 'file_path': 'paddleocr_vl/道通24年年报_p0101-0150/道通24年年报_p0101-0150_42.md', 'chunk_in_file': 4, 'h3': '1. 事项描述', 'parse_source': 'paddleocr_vl', 'h2': '(一) 收入确认', 'doc_group': '道通24年年报_p0101-0150', 'source': '道通24年年报', 'chunk_id': 797, 'file_name': '道通24年年报_p0101-0150_42.md'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'parse_source': 'paddleocr_vl', 'chunk_in_file': 4, 'file_name': '道通24年年报_p0001-0050_9.md', 'chunk_id': 187, 'source': '道通24年年报', 'doc_group': '道通24年年报_p0001-0050', 'doc_id': '道通24年年报_p0001-0050_9', 'file_path': 'paddleocr_vl/道通24年年报_p0001-0050/道通24年年报_p0001-0050_9.md', 'h3': '(一)主要会计数据'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td rowspan="2">主要会计数据</td><td rowspan="2">20

HyDE

In [12]:
# 3) HyDE：先让 LLM 写一个“可能的答案”，再用这段答案做检索

hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。请为用户问题写一段可能出现在年报中的‘假设答案’，用正式书面语，尽量包含可检索的关键词（业务、产品线、收入、分部等）。不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)

hypo = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
print("hypo (first 400 chars):\n", hypo[:400])

hits = vs.similarity_search(hypo, k=5)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


hypo (first 400 chars):
 在道通2024年年报中，公司的主营业务和产品线收入结构得到了详细的阐述。公司继续专注于其核心业务领域，包括汽车诊断设备、汽车电子产品以及相关软件服务。年报中指出，各业务分部的收入贡献如下：

1. **汽车诊断设备**：作为公司的主要收入来源，该产品线在2024年继续保持强劲增长，得益于全球市场对高效诊断解决方案的持续需求。公司在这一领域的技术创新和市场拓展策略显著提升了市场份额。

2. **汽车电子产品**：该分部的收入在2024年实现了稳步增长，主要受益于新产品的推出和现有产品的升级。公司在智能汽车电子领域的投入和研发能力增强了其市场竞争力。

3. **软件服务**：随着数字化转型的加速，软件服务分部的收入增长显著。公司通过提供综合性软件解决方案和云服务，满足了客户对数据分析和远程诊断的需求。

年报中还强调，各业务分部的协同效应显著提升了整体运营效率，推动了公司收入的多元化和可持
[1] {'source': '道通24年年报', 'file_path': 'paddleocr_vl/道通24年年报_p0001-0050/道通24年年报_p0001-0050_14.md', 'doc_group': '道通24年年报_p0001-0050', 'h3': '（一）数字维修业务——第一发展曲线稳健增长', 'parse_source': 'paddleocr_vl', 'chunk_id': 36, 'file_name': '道通24年年报_p0001-0050_14.md', 'doc_id': '道通24年年报_p0001-0050_14', 'chunk_in_file': 1}
面对新能源电动化与生成式 AI 的技术共振，基于海量的汽车诊断数据和智能硬件，公司以汽车综合诊断产品为依托，以研发创新为驱动，深度剖析用户多元化需求，构筑了多层次的数字维修解决方案生态。   报告期内，数字维修业务商业化进程加速。公司数字维修业务实现营业收入30.18亿元，同比增长13.93%。其中，汽车综合诊断产品实现收入12.67亿元；TPMS系列产品实现收入7.06亿元，同比增长32.55%；ADAS标定产品实现收入3.90亿元，同比增长26.98%；软件升级服务实现收入4.46亿元，同比增长24.20%。技

[2]